# Session 7.4: Object Detection - Face Detection

**Objectives:** Face detection with Haar Cascades and introduction to YOLO
**Time:** 90 minutes

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

print(f'OpenCV version: {cv2.__version__}')

## 1. Face Detection with Haar Cascades

In [ ]:
# Load Haar Cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

def detect_faces(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)
    
    for (x, y, w, h) in faces:
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        roi_gray = gray[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(roi_gray)
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(img, (x+ex, y+ey), (x+ex+ew, y+ey+eh), (0, 255, 0), 2)
    
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), len(faces)

print('Haar Cascade loaded successfully')

## 2. Webcam Face Detection (Real-time)

**Note:** Run this cell to start webcam detection. Press 'q' to quit.

In [ ]:
def realtime_face_detection():
    cap = cv2.VideoCapture(0)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        
        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
            cv2.putText(frame, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
        
        cv2.putText(frame, f'Faces: {len(faces)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.imshow('Face Detection', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()

# Uncomment to run:
# realtime_face_detection()

## 3. Introduction to YOLO (You Only Look Once)

YOLO is a state-of-the-art real-time object detection system.

In [ ]:
# YOLO demonstration (requires pre-trained weights)
# Download from: https://pjreddie.com/darknet/yolo/

def load_yolo():
    net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')
    with open('coco.names', 'r') as f:
        classes = [line.strip() for line in f.readlines()]
    layer_names = net.getLayerNames()
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]
    return net, classes, output_layers

def detect_objects_yolo(img_path, net, classes, output_layers):
    img = cv2.imread(img_path)
    height, width, channels = img.shape
    
    blob = cv2.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)
    outs = net.forward(output_layers)
    
    class_ids, confidences, boxes = [], [], []
    
    for out in outs:
        for detection in out:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5:
                center_x, center_y = int(detection[0] * width), int(detection[1] * height)
                w, h = int(detection[2] * width), int(detection[3] * height)
                x, y = int(center_x - w / 2), int(center_y - h / 2)
                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)
    
    indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
    
    for i in range(len(boxes)):
        if i in indexes:
            x, y, w, h = boxes[i]
            label = str(classes[class_ids[i]])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(img, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

print('YOLO functions defined. Download weights to use.')

## 4. Comparison: Haar Cascade vs Deep Learning

In [ ]:
comparison_data = {
    'Method': ['Haar Cascade', 'YOLO', 'Faster R-CNN'],
    'Speed (FPS)': [30, 45, 7],
    'Accuracy': ['Medium', 'High', 'Very High'],
    'Use Case': ['Simple face detection', 'Real-time multi-object', 'Precision tasks']
}

import pandas as pd
df = pd.DataFrame(comparison_data)
print(df)

## Key Takeaways

- **Haar Cascades:** Fast, simple, works for faces
- **YOLO:** Real-time, multiple objects, state-of-the-art
- **IoU & NMS:** Critical for handling overlapping detections
- **Trade-offs:** Speed vs Accuracy

**Session Complete!**